# DecodeRandomResizeShortCropLong

Demonstrates `DecodeRandomResizeShortCropLong` and its multi-crop variant
`DecodeMultiRandomResizeShortCropLong`.

**Operation:** Resize the shortest edge to `size` (preserving aspect ratio),
then crop the long edge to `size`, producing a square `(size, size)` output.

**Key parameters:**
- `size`: int (fixed) or tuple (min, max) for random sampling
- `x_range`, `y_range`: crop position in [0, 1] — 0=left/top, 0.5=center, 1=right/bottom
- `size_mode`: `"per_batch"` or `"per_image"` when size is a tuple
- `seed`: for reproducibility

In [ ]:
LITDATA_VAL_PATH = "s3://visionlab-datasets/imagenet1k/pre-processed/s256-l512-jpgbytes-q100-streaming/val/"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from slipstream import (
    SlipstreamDataset,
    SlipstreamLoader,
    DecodeRandomResizeShortCropLong,
    DecodeMultiRandomResizeShortCropLong,
    DecodeResizeCrop,
    MultiCropPipeline,
    ToTorchImage,
    Normalize,
    IMAGENET_MEAN,
    IMAGENET_STD,
)
from slipstream.decoders.numba_decoder import _compute_resize_short_crop_long_params

dataset = SlipstreamDataset(
    remote_dir=LITDATA_VAL_PATH,
    decode_images=False,
)
print(f"Dataset: {len(dataset):,} samples")

In [ ]:
def show_batch_grid(batches, row_labels=None, col_labels=None, suptitle=None, n_cols=8):
    """Show one or more batches as a grid.
    
    Args:
        batches: single numpy array [B, H, W, 3] or list of arrays.
        row_labels: label per row.
        col_labels: label per column.
    """
    if isinstance(batches, np.ndarray):
        batches = [batches]
    n_rows = len(batches)
    n_cols = min(n_cols, batches[0].shape[0])
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.2 * n_cols, 2.5 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]
    for r, batch in enumerate(batches):
        for c in range(n_cols):
            axes[r, c].imshow(batch[c])
            axes[r, c].axis('off')
            if r == 0 and col_labels:
                axes[r, c].set_title(col_labels[c], fontsize=8)
        if row_labels:
            axes[r, 0].set_ylabel(row_labels[r], fontsize=10, rotation=0, labelpad=50, va='center')
    if suptitle:
        fig.suptitle(suptitle, fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


def show_batch_grid_true_size(batches, row_labels=None, col_labels=None,
                               suptitle=None, n_cols=8):
    """Show batches as a grid preserving actual pixel sizes.
    
    Rows with smaller images get proportionally less vertical space,
    so a 128x128 crop appears visibly smaller than a 224x224 crop.
    
    Args:
        batches: list of numpy arrays [B, H, W, 3] (may differ in H/W per row).
        row_labels: label per row.
        col_labels: label per column.
    """
    if isinstance(batches, np.ndarray):
        batches = [batches]
    n_rows = len(batches)
    n_cols = min(n_cols, batches[0].shape[0])

    row_heights = [batch.shape[1] for batch in batches]
    max_h = max(row_heights)
    height_ratios = [h / max_h for h in row_heights]

    scale = 2.2  # inches per column at max size
    fig_w = scale * n_cols
    fig_h = sum(h / max_h * scale for h in row_heights) + 0.8

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(fig_w, fig_h),
        gridspec_kw={'height_ratios': height_ratios},
    )
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    for r, batch in enumerate(batches):
        for c in range(n_cols):
            axes[r, c].imshow(batch[c])
            axes[r, c].axis('off')
            if r == 0 and col_labels:
                axes[r, c].set_title(col_labels[c], fontsize=8)
        if row_labels:
            axes[r, 0].set_ylabel(row_labels[r], fontsize=10, rotation=0,
                                   labelpad=55, va='center')
    if suptitle:
        fig.suptitle(suptitle, fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


def show_named_crops(named, names, n_cols=8, suptitle=None):
    """Show named crop dict as rows, preserving actual image sizes.
    
    Each row shows one named crop. Images are displayed at their actual pixel
    size so you can see differences between 224x224 and 96x96.
    """
    n_rows = len(names)
    n_cols = min(n_cols, named[names[0]].shape[0])
    
    # Compute per-row heights based on actual image sizes
    row_heights = [named[name].shape[1] for name in names]
    max_h = max(row_heights)
    height_ratios = [h / max_h for h in row_heights]
    
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(2.2 * n_cols, sum(h * 2.5 / max_h for h in row_heights) + 0.5),
        gridspec_kw={'height_ratios': height_ratios},
    )
    if n_rows == 1:
        axes = axes[np.newaxis, :]
    for r, name in enumerate(names):
        imgs = named[name]
        h, w = imgs.shape[1], imgs.shape[2]
        for c in range(n_cols):
            axes[r, c].imshow(imgs[c])
            axes[r, c].axis('off')
        axes[r, 0].set_ylabel(f'{name}\n{w}x{h}', fontsize=9, rotation=0, labelpad=55, va='center')
    if suptitle:
        fig.suptitle(suptitle, fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


def show_variable_size_images(images, titles=None, suptitle=None):
    """Show a list of differently-sized images at their actual pixel sizes.

    Uses gridspec width_ratios so smaller images get narrower columns,
    visually preserving the size difference between e.g. 96x96 and 224x224.
    """
    n = len(images)
    widths = [img.shape[1] for img in images]
    heights = [img.shape[0] for img in images]
    max_w = max(widths)
    max_h = max(heights)
    width_ratios = [w / max_w for w in widths]

    scale = 2.5  # inches for the largest column
    fig_w = sum(w / max_w * scale for w in widths) + 0.3 * n
    fig_h = max_h / max_w * scale + 1.0

    fig, axes = plt.subplots(
        1, n, figsize=(fig_w, fig_h),
        gridspec_kw={'width_ratios': width_ratios},
    )
    if n == 1:
        axes = [axes]
    for i, (ax, img) in enumerate(zip(axes, images)):
        ax.imshow(img)
        ax.axis('off')
        h, w = img.shape[:2]
        title = titles[i] if titles else ''
        ax.set_title(f'{title}\n{w}x{h}', fontsize=9)
    if suptitle:
        fig.suptitle(suptitle, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 1. Basic Usage: Center Crop (Default)

With default `x_range=(0.5, 0.5)` and `y_range=(0.5, 0.5)`, this is equivalent to
`DecodeResizeCrop(resize_size=size, crop_size=size)` — resize shortest edge, center-crop.

In [ ]:
loader = SlipstreamLoader(
    dataset,
    batch_size=8,
    shuffle=False,
    pipelines={'image': [
        DecodeRandomResizeShortCropLong(size=224),
    ]},
    exclude_fields=['path'],
    verbose=False,
)

batch = next(iter(loader))
loader.shutdown()

print(f"Image shape: {batch['image'].shape}, dtype: {batch['image'].dtype}")
show_batch_grid(batch['image'], suptitle='Center crop (default) — all 224x224')

## 2. Equivalence with DecodeResizeCrop

When `x_range=0.5, y_range=0.5` (center crop), the output is pixel-identical
to `DecodeResizeCrop(resize_size=size, crop_size=size)`.

In [ ]:
loader_new = SlipstreamLoader(
    dataset, batch_size=8, shuffle=False,
    pipelines={'image': [DecodeRandomResizeShortCropLong(size=224)]},
    exclude_fields=['path'], verbose=False,
)
loader_ref = SlipstreamLoader(
    dataset, batch_size=8, shuffle=False,
    pipelines={'image': [DecodeResizeCrop(resize_size=224, crop_size=224)]},
    exclude_fields=['path'], verbose=False,
)

batch_new = next(iter(loader_new))
batch_ref = next(iter(loader_ref))
loader_new.shutdown()
loader_ref.shutdown()

match = np.array_equal(batch_new['image'], batch_ref['image'])
print(f"Pixel-identical to DecodeResizeCrop(224, 224): {match}")
if not match:
    diff = np.abs(batch_new['image'].astype(int) - batch_ref['image'].astype(int))
    print(f"Max diff: {diff.max()}, Mean diff: {diff.mean():.4f}")

## 3. Crop Position Control: x_range

`x_range` slides the crop horizontally for landscape images. For portrait images,
there is no horizontal slack so `x_range` has no effect.

In [ ]:
# DecodeRandomResizeShortCropLong?

In [ ]:
positions = [0.0, 0.25, 0.5, 0.75, 1.0]
rows = []
for xp in positions:
    loader = SlipstreamLoader(
        dataset, batch_size=8, shuffle=False,
        pipelines={'image': [DecodeRandomResizeShortCropLong(size=224, x_range=xp)]},
        exclude_fields=['path'], verbose=False,
    )
    b = next(iter(loader))
    rows.append(b['image'])
    loader.shutdown()

show_batch_grid(
    rows,
    row_labels=[f'x={p}' for p in positions],
    suptitle='Sliding x_range from 0 (left) to 1 (right)',
)

## 4. Random Crop Positions

Apply `x_range=(0, 1)` and `y_range=(0, 1)` to the **same images** 3 times
with different seeds. Each row is a different random draw on the same batch.

Subplot titles show: `WxH` (original size), `x=... y=...` (sampled positions),
and `slack` (how many resized pixels of freedom exist on each axis).
- Slack is **consistent across rows** (same images → same dimensions).
- Images with `slack (0,0)` are square — the crop cannot move regardless of x/y.

In [ ]:
# Decode the SAME first batch 3 times with different random seeds.
# A fresh loader per trial ensures we always load images 0-7 (shuffle=False).
seeds = [52, 43, 44]
rows = []
params_list = []

for seed in seeds:
    dec = DecodeRandomResizeShortCropLong(
        size=224, x_range=(0, 1), y_range=(0, 1), seed=seed,
    )
    loader = SlipstreamLoader(
        dataset, batch_size=8, shuffle=False,
        pipelines={'image': [dec]},
        exclude_fields=['path'], verbose=False,
    )
    batch = next(iter(loader))
    rows.append(batch['image'])
    params_list.append(dec.last_params)
    loader.shutdown()

# Plot with params as subplot titles
n_rows = len(rows)
n_cols = min(8, rows[0].shape[0])
fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.5 * n_cols, 3.5 * n_rows))
if n_rows == 1:
    axes = axes[np.newaxis, :]

for r in range(n_rows):
    p = params_list[r]
    for c in range(n_cols):
        axes[r, c].imshow(rows[r][c])
        axes[r, c].axis('off')
        h, w = int(p['heights'][c]), int(p['widths'][c])
        xp, yp = p['x_pos'][c], p['y_pos'][c]
        sz = int(p['target_sizes'][c])
        # Compute slack in resized space
        scale = sz / min(h, w)
        new_h, new_w = int(h * scale + 0.5), int(w * scale + 0.5)
        sx, sy = max(0, new_w - sz), max(0, new_h - sz)
        axes[r, c].set_title(
            f'{w}x{h}\nx={xp:.2f} y={yp:.2f}\nslack ({sx},{sy})',
            fontsize=14, pad=2,
        )
    axes[r, 0].set_ylabel(f'seed={seeds[r]}', fontsize=10, rotation=0,
                           labelpad=45, va='center')

fig.suptitle(
    'Random crops of the SAME images (different seeds)\n'
    'Slack shows available movement — (0,0) = square, no variation possible',
    fontsize=14, fontweight='bold', y=1.02,
)
plt.tight_layout()
plt.show()

## 5. Asymmetric x/y Ranges

`x_range=(0, 1)` with `y_range=(0.5, 0.5)` allows horizontal jitter for
landscape images while keeping portrait images vertically centered.

Same images decoded 3 times with different decoder seeds to show the jitter effect.

In [ ]:
# Same images, 3 different decoder seeds — fresh loader per trial
seeds = [42, 43, 44]
rows = []
for seed in seeds:
    dec = DecodeRandomResizeShortCropLong(
        size=224,
        x_range=(0, 1),       # jitter horizontal for wide images
        y_range=(0.5, 0.5),   # always center-crop vertical
        seed=seed,
    )
    loader = SlipstreamLoader(
        dataset, batch_size=8, shuffle=False,
        pipelines={'image': [dec]},
        exclude_fields=['path'], verbose=False,
    )
    rows.append(next(iter(loader))['image'])
    loader.shutdown()

show_batch_grid(
    rows,
    row_labels=[f'seed={s}' for s in seeds],
    suptitle='Asymmetric: x_range=(0,1), y_range=(0.5,0.5)\nWide images jitter horizontally, tall images stay centered',
)

## 6. Crop Region Visualization

Visualize where crop regions fall on the original image dimensions at
different `x_range` / `y_range` settings. These are overlaid on a
schematic of the original image (not the decoded pixels).

In [ ]:
def plot_crop_region(ax, img_w, img_h, target_size, x_pos, y_pos, color='red', label=''):
    """Overlay crop region rectangle on axes."""
    heights = np.array([img_h], dtype=np.uint32)
    widths = np.array([img_w], dtype=np.uint32)
    target_sizes = np.array([target_size], dtype=np.int32)
    xp = np.array([x_pos], dtype=np.float64)
    yp = np.array([y_pos], dtype=np.float64)
    params = _compute_resize_short_crop_long_params(heights, widths, target_sizes, xp, yp)
    cx, cy, cw, ch = params[0]
    rect = patches.Rectangle((cx, cy), cw, ch, linewidth=2,
                              edgecolor=color, facecolor=color, alpha=0.3)
    ax.add_patch(rect)
    if label:
        ax.annotate(label, (cx + cw/2, cy + ch/2), ha='center', va='center',
                    fontsize=8, color=color, fontweight='bold')


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['blue', 'green', 'red', 'purple', 'orange']

for ax, (img_w, img_h, title) in zip(axes, [
    (800, 400, 'Wide 800x400'),
    (480, 640, 'Portrait 480x640'),
]):
    ax.set_xlim(0, img_w)
    ax.set_ylim(img_h, 0)
    ax.set_aspect('equal')
    ax.add_patch(patches.Rectangle((0, 0), img_w, img_h, linewidth=2,
                                    edgecolor='black', facecolor='lightgray', alpha=0.3))
    for i, pos in enumerate([0.0, 0.25, 0.5, 0.75, 1.0]):
        plot_crop_region(ax, img_w, img_h, 224, pos, pos, colors[i], f'{pos}')
    ax.set_title(f'{title}: crop positions 0.0 \u2192 1.0', fontsize=10, fontweight='bold')
    ax.set_xlabel('x (pixels)')
    ax.set_ylabel('y (pixels)')

plt.tight_layout()
plt.show()

## 7. Random Size (per_batch)

When `size=(min, max)` with `size_mode="per_batch"`, one random size is sampled
per batch. All images in the batch share the same output size.

In [ ]:
loader = SlipstreamLoader(
    dataset, batch_size=8, shuffle=False,
    pipelines={'image': [
        DecodeRandomResizeShortCropLong(size=(128, 256), size_mode='per_batch', seed=42),
    ]},
    exclude_fields=['path'], verbose=False,
)

it = iter(loader)
rows = []
labels = []
for i in range(4):
    b = next(it)
    img = b['image']
    s = img.shape[1]
    rows.append(img)
    labels.append(f'{s}x{s}')
    print(f"Batch {i}: size={s}x{s}")
loader.shutdown()

show_batch_grid_true_size(rows, row_labels=labels,
                           suptitle='Per-batch random size (128-256) — true pixel sizes')

## 8. Random Size (per_image)

When `size_mode="per_image"`, each image gets its own random size.
Returns a list of differently-sized arrays.

Images are displayed at their **actual pixel sizes** — a 96x96 image
appears smaller than a 224x224 image.

In [ ]:
loader = SlipstreamLoader(
    dataset, batch_size=6, shuffle=False,
    pipelines={'image': [
        DecodeRandomResizeShortCropLong(size=(96, 256), size_mode='per_image', seed=42),
    ]},
    exclude_fields=['path'], verbose=False,
)

batch = next(iter(loader))
loader.shutdown()

images = batch['image']
print(f"Type: {type(images)}, Length: {len(images)}")
for i, arr in enumerate(images):
    print(f"  Image {i}: {arr.shape}")

show_variable_size_images(
    images,
    titles=[f'img {i}' for i in range(len(images))],
    suptitle='Per-image random size (96-256): displayed at actual pixel sizes',
)

## 9. Multi-Crop Variant

`DecodeMultiRandomResizeShortCropLong` decodes each JPEG once, then applies
multiple named crops at different sizes. Returns a dict of named arrays.

In [ ]:
view_params = {
    'large': dict(size=224, x_range=(0, 1), y_range=(0, 1), seed=42),
    'medium': dict(size=160, x_range=(0, 1), y_range=(0, 1), seed=42),
    'small': dict(size=96, x_range=(0, 1), y_range=(0, 1), seed=42),
}
view_names = list(view_params.keys())

loader = SlipstreamLoader(
    dataset, batch_size=8, shuffle=False,
    pipelines={'image': [
        DecodeMultiRandomResizeShortCropLong(view_params),
    ]},
    exclude_fields=['path'], verbose=False,
)

batch = next(iter(loader))
loader.shutdown()

for name in view_names:
    print(f"  {name}: {batch[name].shape}")

show_named_crops(batch, view_names,
                 suptitle='Multi-crop: 3 named crops at different sizes (actual pixel sizes)')

## 10. Yoked Multi-Crops (Same Seed)

Crops with the **same seed** produce the same crop position, even at different
sizes. This is useful for multi-scale SSL where you want views of the same
region at different resolutions.

In [ ]:
# Yoked: same seed → same crop position
loader_yoked = SlipstreamLoader(
    dataset, batch_size=8, shuffle=False,
    pipelines={'image': [
        DecodeMultiRandomResizeShortCropLong({
            'large': dict(size=224, x_range=(0, 1), y_range=(0, 1), seed=42),
            'small': dict(size=96, x_range=(0, 1), y_range=(0, 1), seed=42),  # Same seed!
        }),
    ]},
    exclude_fields=['path'], verbose=False,
)

# Independent: different seeds → different crop positions
loader_indep = SlipstreamLoader(
    dataset, batch_size=8, shuffle=False,
    pipelines={'image': [
        DecodeMultiRandomResizeShortCropLong({
            'large': dict(size=224, x_range=(0, 1), y_range=(0, 1), seed=42),
            'small': dict(size=96, x_range=(0, 1), y_range=(0, 1), seed=99),  # Different seed!
        }),
    ]},
    exclude_fields=['path'], verbose=False,
)

yoked = next(iter(loader_yoked))
indep = next(iter(loader_indep))
loader_yoked.shutdown()
loader_indep.shutdown()

# True-size grid: 224 columns are wider/taller than 96 columns
fig, axes = plt.subplots(
    2, 2, figsize=(8, 5),
    gridspec_kw={'width_ratios': [224, 96], 'height_ratios': [224, 96]},
)

for ax_row, (data, label_prefix) in zip(axes, [(yoked, 'Yoked'), (indep, 'Independent')]):
    for ax, (name, sz) in zip(ax_row, [('large', 224), ('small', 96)]):
        ax.imshow(data[name][0])
        seed_label = 'seed=42' if (label_prefix == 'Yoked' or name == 'large') else 'seed=99'
        ax.set_title(f'{label_prefix}: {name} {sz}x{sz} ({seed_label})', fontsize=10)
        ax.axis('off')

fig.suptitle('Yoked (top): same region at different scales\nIndependent (bottom): different regions',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Multi-Crop with Pipeline Transforms

Combine `DecodeMultiRandomResizeShortCropLong` with `MultiCropPipeline` for
per-view transforms (normalization, augmentation, etc.).

In [ ]:
DEVICE = 'cpu'

loader = SlipstreamLoader(
    dataset, batch_size=8, shuffle=True, seed=42,
    pipelines={'image': [
        DecodeMultiRandomResizeShortCropLong({
            'global_view': dict(size=224, x_range=(0, 1), y_range=(0, 1), seed=42),
            'local_view': dict(size=96, x_range=(0, 1), y_range=(0, 1), seed=43),
        }),
        MultiCropPipeline({
            'global_view': [
                ToTorchImage(device=DEVICE),
                Normalize(IMAGENET_MEAN, IMAGENET_STD, device=DEVICE),
            ],
            'local_view': [
                ToTorchImage(device=DEVICE),
                Normalize(IMAGENET_MEAN, IMAGENET_STD, device=DEVICE),
            ],
        }),
    ]},
    exclude_fields=['path'], verbose=False,
)

batch = next(iter(loader))
loader.shutdown()

# Multi-crop results are top-level batch keys (not nested under 'image')
print(f"global_view: {batch['global_view'].shape}, dtype={batch['global_view'].dtype}")
print(f"  mean={batch['global_view'].mean():.4f}, std={batch['global_view'].std():.4f}")
print(f"local_view:  {batch['local_view'].shape}, dtype={batch['local_view'].dtype}")
print(f"  mean={batch['local_view'].mean():.4f}, std={batch['local_view'].std():.4f}")
print(f"label: {batch['label'].shape}")

## Summary

| Class | Output | Use case |
|-------|--------|----------|
| `DecodeRandomResizeShortCropLong` | Single array `[B, s, s, 3]` | Evaluation, simple augmentation |
| `DecodeMultiRandomResizeShortCropLong` | Dict of named arrays | Multi-scale SSL, paired views |

| Parameter | Effect |
|-----------|--------|
| `size=224` | Fixed 224x224 output |
| `size=(128, 256)` | Random size per batch or per image |
| `x_range=(0, 1)` | Full horizontal jitter (wide images) |
| `y_range=(0.5, 0.5)` | Always center vertically (tall images) |
| `seed=42` | Reproducible crops; same seed = yoked crops |